# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a ranking task. The goal is to assign each content item a priority that determines the order in which a reviewer should inspect them. The output is a ranked queue ordered by predicted need for attention.

In [4]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print("Trend direction distribution:")
print(df["trend_direction"].value_counts())
print(f"\nTotal pages: {len(df)}")
print(f"Declining pages: {(df['trend_direction'] == 'down').sum()} ({(df['trend_direction'] == 'down').mean():.1}")

Trend direction distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Total pages: 30000
Declining pages: 16262 (0.5


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The target is: is_declining_label = (trend_direction == "down")

It is a proxy label defined from the current 90-day window's trend direction.

In [5]:
import pandas as pd

label = (df["trend_direction"] == "down").astype(int)
print(f"Label: is_declining_label = (trend_direction == 'down')")
print(f"Label rate: {label.mean():.3f}")
print(f"\nLabel balance:")
print(label.value_counts().rename({0: "not declining (0)", 1: "declining (1)"}))


Label: is_declining_label = (trend_direction == 'down')
Label rate: 0.542

Label balance:
trend_direction
declining (1)        16262
not declining (0)    13738
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@50

This metric asks: of the top 50 pages the system recommends for review, how many are actually declining? It matches the real decision because content teams have limited capacity, they will not review all 30,000 pages. They will start at the top of the queue and work down until they run out of time. Precision@50 directly measures how useful the first 50 recommendations are.

In [6]:
import pandas as pd
import numpy as np

np.random.seed(42)
random_sample_labels = label.sample(50).values
random_p50 = random_sample_labels.mean()
print(f"Random Precision@50 (single run): {random_p50:.3f}")

n_trials = 10000
random_p50s = [label.sample(50).values.mean() for _ in range(n_trials)]
print(f"Random Precision@50 (avg over {n_trials} runs): {np.mean(random_p50s):.3f}")
print(f"Label base rate: {label.mean():.3f}")


print(f"\nCommitted baseline Precision@50: 0.240")
print(f"Committed random forest Precision@50: 0.740")
print(f"\nA useful model should clearly beat the random baseline of ~{label.mean():.2f}")

Random Precision@50 (single run): 0.620
Random Precision@50 (avg over 10000 runs): 0.542
Label base rate: 0.542

Committed baseline Precision@50: 0.240
Committed random forest Precision@50: 0.740

A useful model should clearly beat the random baseline of ~0.54


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (page). There are 30,000 content items from 32 clients. Each row contains observed search and engagement metrics aggregated over a trailing 90-day window, plus content metadata and derived tiers. The unit is the page, not the client, not the query, and not a time point. This matters because pages from the same client may share patterns, the train/test split must hold out entire clients, not random rows, to test whether the model generalizes to unseen clients.

In [7]:
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Unique clients: {df['client_id'].nunique()}")
print(f"Unique content items: {df['content_id'].nunique()}")
print(f"\nOne row = one content item (page)")
print(f"Each row has trailing-90-day metrics + content metadata + derived tiers")

key_cols = [
    "content_id", "client_id", "impressions_90d", "clicks_90d",
    "sessions_90d", "ctr", "avg_position", "content_age_days",
    "trend_direction", "word_count"
]
print(f"\nSample rows (key columns):")
df[key_cols].head(8)

Shape: 30000 rows × 44 columns
Unique clients: 32
Unique content items: 30000

One row = one content item (page)
Each row has trailing-90-day metrics + content metadata + derived tiers

Sample rows (key columns):


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,trend_direction,word_count
0,content_304f48230142,client_f369cb89fc,3803,29,17,0.76,10.6,187,down,3221.0
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,0.05,20.3,445,down,2481.0
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,0.09,36.5,141,down,3515.0
3,content_331d6c4de07b,client_19581e27de,11751,58,78,0.49,6.2,463,stable,NaN
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,0.13,44.0,263,down,2803.0
5,content_d4084a4bc775,client_f369cb89fc,3970,1,5,0.03,8.5,147,down,3080.0
6,content_9a34b442b552,client_8722616204,20,0,1,0.00,7.0,90,down,3059.0
7,content_a63219c6e95a,client_19581e27de,1724,1,28,0.06,21.2,445,stable,NaN


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can flag obvious cases: stale visible pages, thin content, declining pages with enough demand. It finds some real opportunities, but most of its top-50 picks are false positives. The reason is that the "right page to fix" is not defined by any
single signal. It is the intersection of many signals: enough impressions to matter, declining or weak CTR relative to position, low engagement relative to sessions, content age, freshness, word count, and the shape of the trend. A human could write a longer rule, but the interactions are non-linear and the thresholds shift depending on the content type and position tier. A learned model can weight these signals jointly and find combinations that a hand-written rule misses. The committed random forest achieves a Precision@50 of 0.740 — roughly 3x the baseline — which is strong evidence
that the pattern is real but too complex for a single if-statement.

In [ ]:
print("Decline rate by impression tier:")
df["impression_tier"] = pd.cut(
    df["impressions_90d"],
    bins=[0, 1, 100, 300, 3000, 30000, float("inf")],
    labels=["none", "low", "moderate", "good", "high", "excellent"],
    right=True
)
print(df.groupby("impression_tier", observed=True).apply(
    lambda g: (g["trend_direction"] == "down").mean()
).round(3))

print("\nDecline rate by position tier:")
print(df.groupby("position_tier", observed=True).apply(
    lambda g: (g["trend_direction"] == "down").mean()
).round(3))

print("\n--- Baseline vs Model (from committed results) ---")
print(f"Baseline (hand rules)   Precision@50: 0.240")
print(f"Random forest (learned) Precision@50: 0.740")
print(f"Lift: ~3x — the pattern is real but too tangled for if-statements.")

Decline rate by impression tier:
impression_tier
none         0.084
low          0.437
moderate     0.614
good         0.615
high         0.586
excellent    0.462
dtype: float64

Decline rate by position tier:
position_tier
deep        0.344
page_1      0.570
page_3_5    0.562
striking    0.610
top_3       0.241
dtype: float64

--- Baseline vs Model (from committed results) ---
Baseline (hand rules)   Precision@50: 0.240
Random forest (learned) Precision@50: 0.740
Lift: ~3x — the pattern is real but too tangled for if-statements.


C:\Users\Admin\AppData\Local\Temp\ipykernel_8020\727665649.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby("impression_tier", observed=True).apply(
C:\Users\Admin\AppData\Local\Temp\ipykernel_8020\727665649.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(df.groupby("position_tier", observed=True).apply(


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.